# 5. Results

Collects the outputs of steps 2 to 4. Every intervention is compared with its control on the same 15 seeds; the tables give the mean paired change with a 95% confidence interval, and whether it survives a Bonferroni correction for the 8 tests.

In [ ]:
import glob, math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

def one(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True), key=len)[0]

def ci(x, conf=0.95):
    x = np.asarray(x, float)
    return x.mean(), stats.t.ppf(0.5 + conf / 2, len(x) - 1) * x.std(ddof=1) / math.sqrt(len(x))

h = pd.concat([pd.read_csv(f) for f in glob.glob("/kaggle/input/**/interventions_seeds*.csv", recursive=True)])
p = pd.concat([pd.read_csv(f) for f in glob.glob("/kaggle/input/**/projection_seeds*.csv", recursive=True)])
print(len(h), "intervention runs,", len(p), "projection runs")

## Validation

In [ ]:
s = pd.read_csv(one("results/jica_sweep_summary.csv"))
m = s[s.config == "main_y0.2"]
for k, label in [("ring_kmh", "Ring Road km/h (survey 24.0)"), ("inner_kmh", "inner main roads km/h (survey 15.3)"),
                 ("share", "model / counted volume, 9 junctions"), ("corr", "correlation across junctions"),
                 ("within_1_5", "junctions within a factor 1.5"), ("geh_pass", "junctions with GEH < 5")]:
    mean, half = ci(m[k]); print(f"{label:<40} {mean:7.2f} +/- {half:.2f}")

geh = pd.concat([pd.read_csv(f) for f in glob.glob("/kaggle/input/**/results/jica_sweep/_geh_main_y0.2_s*.csv", recursive=True)])
r = (geh.model_pcu_h / geh.observed_pcu_h).groupby(geh.intersection).agg(["mean", "std", "count"]).sort_values("mean")
ax = r["mean"].plot.barh(xerr=r["std"] / np.sqrt(r["count"]) * 2.145, figsize=(6, 4), color="#2a5a8c")
ax.axvline(1, color="k"); ax.axvspan(1 / 1.5, 1.5, color="green", alpha=0.1)
ax.set_xlabel("model / observed peak entering volume"); plt.tight_layout(); plt.show()

## Interventions (H1 to H7)

In [ ]:
tests = [("H1 adaptive signals", "adaptive", "baseline"), ("H2 congestion pricing", "pricing", "baseline"),
         ("H3 adaptive + pricing", "adaptive_pricing", "baseline"), ("H4 tidal lanes", "am_tidal", "am_no_tidal"),
         ("H5 parking enforcement", "parking", "baseline"), ("H6 staggered departures", "staggered", "baseline"),
         ("H7 decentralized", "decentralized", "baseline"), ("H7 vs equal uniform cut", "decentralized", "uniform_cut")]
w = {k: h.pivot(index="seed", columns="config", values=k) for k in ("tis_s", "delay_s", "completed", "trips")}
conf_b = 1 - 0.05 / len(tests)
rows = []
for name, c, b in tests:
    rel = (w["tis_s"][c] - w["tis_s"][b]) / w["tis_s"][b] * 100
    dl = (w["delay_s"][c] - w["delay_s"][b]) / w["delay_s"][b] * 100
    t, th = ci(rel); d, dh = ci(dl); cm, chh = ci((w["completed"][c] - w["completed"][b]) * 100)
    rows.append({"test": name, "time in system %": f"{t:+.1f} +/- {th:.1f}", "delay %": f"{d:+.1f} +/- {dh:.1f}",
                 "completion pts": f"{cm:+.1f} +/- {chh:.1f}", "seeds better": f"{(rel < 0).sum()}/{len(rel)}",
                 "Bonferroni": "yes" if abs(t) > ci(rel, conf_b)[1] else "no",
                 "vehicles": f"{int(w['trips'][c].iloc[0]):,} vs {int(w['trips'][b].iloc[0]):,}",
                 "_m": t, "_h": th})
res = pd.DataFrame(rows)
display(res.drop(columns=["_m", "_h"]))
x = w["tis_s"]
inter = (x.adaptive_pricing - x.adaptive - x.pricing + x.baseline) / x.baseline * 100
im, ih = ci(inter)
print(f"H3 interaction (combined minus both single effects, % of baseline): {im:+.1f} +/- {ih:.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.barh(res.test, res._m, xerr=res._h, capsize=4, color=["#2a5a8c" if abs(m) > e else "#999" for m, e in zip(res._m, res._h)])
ax.axvline(0, color="k"); ax.invert_yaxis(); ax.set_xlabel("change in mean time in system (%), 15 paired seeds")
plt.tight_layout(); plt.show()

## Demand growth projection

In [ ]:
pw = {k: p.pivot(index="seed", columns="config", values=k) for k in ("tis_s", "completed", "trips")}
for c in ["baseline", "x1.20_4yr", "x1.94_2026", "x2.32_2030"]:
    comp, ch = ci(pw["completed"][c] * 100)
    line = f"{c:<12} {int(pw['trips'][c].iloc[0]):>8,} vehicles   completed {comp:5.1f}% +/- {ch:.1f}"
    if c != "baseline":
        t, th = ci((pw["tis_s"][c] - pw["tis_s"].baseline) / pw["tis_s"].baseline * 100)
        line += f"   time in system {t:+.0f}% +/- {th:.0f}"
    print(line)